# Adaptive RAG — Complete Implementation (LangChain + LangGraph, 2026 stack)

This notebook implements **Adaptive RAG** on a company-policy knowledge base (`company_docs/*.txt`).

**Pipeline:** Router → Retrieve → Grade documents → Generate → Grade hallucination → Grade answer,
with a **Question Rewriter** loop and a **web-search fallback** for out-of-scope questions.

All imports use the current LangChain ecosystem — no legacy `langchain` chains, no `langchain_experimental`.

## 1. Setup

```bash
pip install -U langchain langchain-core langchain-openai langchain-community langchain-text-splitters langgraph faiss-cpu tavily-python
```

Set your keys:
- `OPENAI_API_KEY` — LLM + embeddings (any OpenAI-compatible endpoint works too)
- `TAVILY_API_KEY` — only needed for the web-search fallback node

In [ ]:
import os
os.environ.setdefault("OPENAI_API_KEY", "sk-your-key-here")
os.environ.setdefault("TAVILY_API_KEY", "tvly-your-key-here")  # optional fallback

# ---- Current (2026) imports only ----
from typing import Literal, Annotated, TypedDict

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from pprint import pprint

# ---- Model & embeddings ----
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 2. Load the knowledge base and build the vector store

The five company policy files are chunked and embedded into a local FAISS index.

In [ ]:
from pathlib import Path

KB_DIR = Path("company_docs")  # put the 5 .txt files here (next to this notebook)

docs = [
    Document(page_content=p.read_text(encoding="utf-8"),
             metadata={"source": p.name})
    for p in sorted(KB_DIR.glob("*.txt"))
]
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## 3. The graders — structured LLM decisions

Adaptive RAG is built from small, single-purpose LLM calls. Each returns a typed
decision (`Literal[...]`) so the graph can branch on it.

In [ ]:
# ---------- 3a. ROUTER: where should this question go? ----------
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert router for a question-answering system.
Classify the question into exactly one of:
- "vectorstore": answerable from internal company policy documents
  (leave, work-from-home, travel, handbook, insurance topics)
- "websearch": requires current/external real-world information
  (news, sports, weather, current events, anything outside company policy)
- "direct": general knowledge or casual chat answerable without retrieval

Return ONLY one word: vectorstore, websearch, or direct."""),
    ("human", "{question}"),
])
structured_router = router_prompt | llm
# NOTE: no .with_structured_output needed — we parse the single-word reply below.

def route_question(state) -> Literal["vectorstore", "websearch", "direct"]:
    decision = structured_router.invoke({"question": state["question"]}).content.strip().lower()
    for key in ("vectorstore", "websearch", "direct"):
        if key in decision:
            return key
    return "vectorstore"  # safe default

# ---------- 3b. RETRIEVAL GRADER: are these documents relevant? ----------
retrieval_grader_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a grader assessing relevance of a retrieved document to a user question.
If the document contains keyword(s) or semantic meaning related to the question, grade it relevant.
Give a binary score: "yes" or "no"."""),
    ("human", "Document:\n\n{document}\n\nQuestion: {question}"),
])
retrieval_grader = retrieval_grader_prompt | llm | StrOutputParser()

# ---------- 3c. HALLUCINATION GRADER: is the answer grounded? ----------
hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a grader assessing whether an answer is grounded in / supported by a set of facts.
Give a binary score: "yes" (supported) or "no" (not supported)."""),
    ("human", "Facts:\n\n{documents}\n\nAnswer: {generation}"),
])
hallucination_grader = hallucination_prompt | llm | StrOutputParser()

# ---------- 3d. ANSWER GRADER: does the answer resolve the question? ----------
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a grader assessing whether an answer resolves a question.
Give a binary score: "yes" (useful) or "no" (not useful)."""),
    ("human", "Question: {question}\n\nAnswer: {generation}"),
])
answer_grader = answer_prompt | llm | StrOutputParser()

# ---------- 3e. QUESTION REWRITER: clarify & retry retrieval ----------
rewriter_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a question re-writer that converts an input question into a better version
optimized for vectorstore retrieval. Look at the input and reason about the underlying semantic
intent. Keep it concise. Return ONLY the improved question."""),
    ("human", "Here is the initial question:\n\n{question}\n\nFormulate an improved question."),
])
question_rewriter = rewriter_prompt | llm | StrOutputParser()

# ---------- 3f. GENERATOR: the actual RAG answer ----------
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an assistant for company policy questions. Use the following retrieved
context to answer the question. If the context does not contain the answer, say you do not know.
Keep the answer concise (3-5 sentences maximum).\n\nContext:\n{context}"""),
    ("human", "{question}"),
])
def format_docs(docs):
    return "\n\n".join(d.text if hasattr(d, "text") else d.page_content for d in docs)

rag_chain = rag_prompt | llm | StrOutputParser()

## 4. Tools, state, and graph nodes

In [ ]:
# ---------- 4a. Web search tool (fallback for out-of-scope questions) ----------
web_search_tool = TavilySearchResults(max_results=3)

# ---------- 4b. State ----------
class AdaptiveRAGState(TypedDict):
    question: str
    documents: list            # current graded documents
    generation: str
    rewrite_tries: int
    max_rewrites: int
    web_search_done: bool

# ---------- 4c. Nodes ----------
def retrieve(state: AdaptiveRAGState):
    """Vector-store retrieval — replaces irrelevant docs with freshly graded ones."""
    docs = retriever.invoke(state["question"])
    filtered = [
        d for d in docs
        if "yes" in retrieval_grader.invoke(
            {"document": d.page_content, "question": state["question"]}
        ).lower()
    ]
    if filtered:                       # found relevant docs -> keep them
        return {"documents": filtered}
    return {"documents": [], "rewrite_tries": state.get("rewrite_tries", 0)}

def rewrite_or_fallback(state: AdaptiveRAGState):
    """No relevant docs after retrieval -> rewrite the question for one more try,
    else fall back to web search."""
    if state.get("rewrite_tries", 0) < state.get("max_rewrites", 2):
        better = question_rewriter.invoke({"question": state["question"]}).strip()
        return {"question": better, "rewrite_tries": state.get("rewrite_tries", 0) + 1}
    return {"web_search_done": True}   # signal: exhausted, go to web

def web_search(state: AdaptiveRAGState):
    """Search the web and store results as documents."""
    results = web_search_tool.invoke({"query": state["question"]})
    docs = [Document(page_content=r["content"], metadata={"source": r["url"]})
            for r in results]
    return {"documents": docs, "generation": ""}   # blank -> will route to generate

def direct_answer(state: AdaptiveRAGState):
    """Answer general-knowledge questions with no retrieval at all."""
    ans = llm.invoke([
        {"role": "system", "content": "Answer concisely and helpfully."},
        {"role": "user", "content": state["question"]},
    ]).content
    return {"generation": ans}

def generate(state: AdaptiveRAGState):
    """RAG generation over the current (graded) documents."""
    generation = rag_chain.invoke({
        "context": format_docs(state["documents"]),
        "question": state["question"],
    })
    return {"generation": generation}

def decide_after_generation(state: AdaptiveRAGState) -> Literal["useful", "not_useful", "not_supported"]:
    """Double quality gate: hallucination check first, then usefulness check."""
    if "yes" not in hallucination_grader.invoke({
        "documents": format_docs(state["documents"]),
        "generation": state["generation"],
    }).lower():
        return "not_supported"
    if "yes" in answer_grader.invoke({
        "question": state["question"], "generation": state["generation"],
    }).lower():
        return "useful"
    return "not_useful"

## 5. Assemble and compile the graph

In [ ]:
builder = StateGraph(AdaptiveRAGState)

builder.add_node("retrieve", retrieve)
builder.add_node("rewrite_or_fallback", rewrite_or_fallback)
builder.add_node("web_search", web_search)
builder.add_node("direct_answer", direct_answer)
builder.add_node("generate", generate)

# Router as the entry branching edge
def route_entry(state: AdaptiveRAGState):
    return route_question(state)

# Replace placeholder: entry edges
builder.add_conditional_edges(START, route_entry, {
    "vectorstore": "retrieve",
    "websearch": "web_search",
    "direct": "direct_answer",
})

builder.add_edge("direct_answer", END)
builder.add_edge("web_search", "generate")

# After retrieval: docs found -> generate; none -> rewrite / fallback
def after_retrieval(state: AdaptiveRAGState):
    return "generate" if state.get("documents") else "rewrite_or_fallback"

builder.add_conditional_edges("retrieve", after_retrieval, {
    "generate": "generate",
    "rewrite_or_fallback": "rewrite_or_fallback",
})

# After rewrite: retry retrieval if we rewrote, else go to web search
def after_rewrite(state: AdaptiveRAGState):
    return "web_search" if state.get("web_search_done") else "retrieve"

builder.add_conditional_edges("rewrite_or_fallback", after_rewrite, {
    "retrieve": "retrieve",
    "web_search": "web_search",
})

# After generation: quality gates
builder.add_conditional_edges("generate", decide_after_generation, {
    "useful": END,
    "not_supported": "generate",          # regenerate (loop guard: docs are stable)
    "not_useful": "rewrite_or_fallback",  # rethink the question
})

graph = builder.compile()
print("Graph compiled.")
graph

### Inspect the graph

In [ ]:
# Uncomment to render the graph diagram inside the notebook:
# graph.get_graph().draw_mermaid_png()

## 6. Run it — three scenarios

### Scenario A: In-scope question (the happy path)
Router → vectorstore → retrieve → relevant docs → generate → useful → **END**

In [ ]:
state_a = graph.invoke({
    "question": "How many days of paid annual leave do I get?",
    "documents": [], "generation": "", "rewrite_tries": 0,
    "max_rewrites": 2, "web_search_done": False,
})
print("ANSWER:\n", state_a["generation"])

### Scenario B: Out-of-scope question (web fallback)
Company policy KB has no answer → retrieval graded irrelevant → rewriter tries twice →
falls back to **web search** → answer grounded in search results.

In [ ]:
state_b = graph.invoke({
    "question": "Who won the FIFA World Cup 2026?",
    "documents": [], "generation": "", "rewrite_tries": 0,
    "max_rewrites": 2, "web_search_done": False,
})
print("ANSWER:\n", state_b["generation"])
pprint(state_b["documents"][:2])

### Scenario C: Ambiguous question (rewriter rescues retrieval)
A vague/ambiguous phrasing retrieves nothing relevant → question rewritten into a
precise retrieval query → second retrieval succeeds → generation passes both gates.

In [ ]:
state_c = graph.invoke({
    "question": "money for working from home",
    "documents": [], "generation": "", "rewrite_tries": 0,
    "max_rewrites": 2, "web_search_done": False,
})
print("ANSWER:\n", state_c["generation"])

## 7. What's next (production hardening ideas)

1. **Structured outputs** — replace word-parsing with `llm.with_structured_output(PydanticModel)` for the graders.
2. **Recursion limits** — pass `graph.invoke(..., config={"recursion_limit": 10})` to cap loops.
3. **Tracing** — add LangSmith (`LANGSMITH_API_KEY`) to visualize every branch decision.
4. **Memory** — persist `documents`/`generation` across turns by swapping `MessagesState`-style accumulation for a checkpointer (`MemorySaver`).
5. **Smarter fallback** — route to a human-support node instead of web search for sensitive policy questions.
